# Day 3: Delta transactions and safe corrections

**Modern Data Engineering for AI Systems · SDA-DSC-214**

Apply a revision-aware fare correction to your Silver table. Check an intentional write rejection and explore history, schema changes and recovery on isolated copies.

**Your lab:** 45 minutes to run and check, 5 minutes to answer, 5 minutes to discuss, and 5 minutes to save.

[Today's guide](https://github.com/almiyead-rgb/masar-ai-data-engineering-ksu/blob/main/day03/README.md) · [Expected results](https://github.com/almiyead-rgb/masar-ai-data-engineering-ksu/tree/main/expected) · [Submission guide](https://github.com/almiyead-rgb/masar-ai-data-engineering-ksu/blob/main/submission/README.md)

## Before you run

Use a CPU runtime. Select the documented Python 3.12 Colab runtime before running the first cell. The setup checks the versions and prepares Java 17. If Colab asks to restart after package installation, reconnect and run the setup again. Keep one GitHub repository for all five days.

Run cells in order. A dependency check is not a completed lab. Compare the outputs produced below with the reference results after the real engines run.

In [ ]:
from pathlib import Path
import importlib.util, json, os, runpy, subprocess, sys
DAY = 3
# Keep your own fork's owner/name here if you want to load committed edits.
PROJECT_REPOSITORY = "almiyead-rgb/masar-ai-data-engineering-ksu"
NONINTERACTIVE = os.environ.get("MASAR_NONINTERACTIVE") == "1"
try:
    IS_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IS_COLAB = False
if IS_COLAB and not NONINTERACTIVE:
    ROOT = Path("/content/masar-project")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/" + PROJECT_REPOSITORY + ".git", str(ROOT)], check=True)
    if not (ROOT / "course.json").is_file():
        raise RuntimeError("The project folder is incomplete. Preserve it and inspect the setup guide.")
    os.chdir(ROOT)
else:
    ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "course.json").is_file() and (p / "src/masar").is_dir()), None)
    if ROOT is None:
        raise FileNotFoundError("Open this notebook from the complete course repository.")
sys.path.insert(0, str(ROOT / "src"))
bootstrap = runpy.run_path(str(ROOT / "scripts/bootstrap_colab.py"))
environment = bootstrap["prepare"](install=IS_COLAB)
os.environ["JAVA_HOME"] = environment["java_home"]
print(json.dumps(environment, indent=2))
from masar.runtime import start_spark
from masar.workspace import require_fixed_dataset, completed_bronze_workspace, new_workspace, write_json
from masar.native_contracts import validate_stage_result
from masar.colab import restore_handoff, record_reflection, record_metrics, save_day, initialize_student_notes
SOURCE = ROOT / "data/masar-small-v1"
require_fixed_dataset(SOURCE)
print("Course source:", ROOT)
print("Dependency checks are complete. The following cells execute the real engines.")
print("Interactive Drive/download UI enabled:", IS_COLAB and not NONINTERACTIVE)


## Your saved project

Allow Colab to mount your own Drive. Your daily continuation archives are saved in `MyDrive/Masar_Project`. A new session restores the preceding day only when the local project has no saved workspace. Existing work is not overwritten.

If you uploaded a handoff ZIP manually, call `restore_handoff` with its path and the preceding day number. Keep earlier archives while you learn.

After restoration, this cell creates your lab-note prompts under `student/dayNN/` and the shared project notes under `student/`. Open the printed paths in Colab’s Files panel and add your own evidence. Existing files are preserved. Your daily question answer belongs in `student/dayNN/REFLECTION.md`; the final save cell includes all these files in both packages.

In [ ]:
DRIVE_FOLDER = None
if IS_COLAB and not NONINTERACTIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_FOLDER = Path("/content/drive/MyDrive/Masar_Project")
    DRIVE_FOLDER.mkdir(parents=True, exist_ok=True)
if DAY > 1:
    pointer = ROOT / "outputs/day01_bronze_success.json"
    if not pointer.exists():
        backup = DRIVE_FOLDER / f"day{DAY-1:02d}_handoff.zip" if DRIVE_FOLDER else None
        if backup is None or not backup.is_file():
            raise FileNotFoundError(
                f"Restore your day{DAY-1:02d}_handoff.zip before continuing. "
                "Use restore_handoff(ROOT, Path('/path/to/your/archive.zip'), expected_day=DAY-1).")
        print(json.dumps(restore_handoff(ROOT, backup, expected_day=DAY-1), indent=2))
    WORK = completed_bronze_workspace(ROOT)
    print("Continue your project:", WORK.relative_to(ROOT))
else:
    pointer = ROOT / "outputs/day01_bronze_success.json"
    if pointer.exists():
        existing = completed_bronze_workspace(ROOT)
        if (existing / "reports/day02_silver.json").exists():
            raise RuntimeError("Your project has advanced beyond Day 1. Open the next day's notebook. Use a separate clean project for a complete replay.")
        WORK = existing
        REUSE_BRONZE = True
        print("Reusing your completed Bronze workspace without re-ingesting the same batches.")
    else:
        WORK = new_workspace(ROOT, "day01_bronze")
        REUSE_BRONZE = False
        print("New project:", WORK.relative_to(ROOT))

# Create prompts only after restoring your existing work. Never overwrite a note.
notes = initialize_student_notes(ROOT, DAY)
print(json.dumps(notes, indent=2))


## Five-minute activity

Decide what to do with a newer approved revision, an identical replay, and a conflicting payload with the same revision.

<details><summary>Discuss the answer after trying</summary>

Accept the newer revision, preserve the identical replay without changing business content, and reject the same-revision conflict for review.

</details>

## Lab 04a: transactions and approved revisions

Apply the correction, replay it, reject an older revision, and inspect the prior Delta version. A same-revision conflict must stop explicitly. Expected final total: **SAR 1,880.60 for 75 trips**.

In [ ]:
from masar.delta_lab import run_transactions_lab
spark = start_spark(WORK, kafka=False)
try:
    result = run_transactions_lab(spark, SOURCE, WORK)
    validate_stage_result('lab04a_transactions', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
    # Observed learning output
    print(json.dumps({k: result[k] for k in ('before','after_correction','past_version_read')}, indent=2, default=str))
    # Your turn: inspect the corrected trip instead of adding the fare difference.
    from pyspark.sql import functions as F
    corrected = spark.read.format("delta").load(str(WORK / "mini_lakehouse/silver/trips"))
    trip_to_inspect = "SYN_T0001"
    observed_fare = corrected.where(F.col("trip_id") == trip_to_inspect).select("fare_sar").first()[0]
    total = corrected.agg(F.sum("fare_sar").alias("total")).first().total
    metrics = {"silver_trips": corrected.count(), "total_fare_sar": format(total, ".2f"),
               "corrected_fare_sar": format(observed_fare, ".2f")}
    print("Corrected trip:", trip_to_inspect, "fare SAR:", observed_fare)
finally:
    spark.stop()

## Lab 04b: safe maintenance and recovery

The schema, compaction and restore exercises use isolated copies. The continuing Silver table remains corrected. VACUUM is a dry run with the retention threshold preserved. An intended rejection is successful only when the correct error is detected and the committed table remains unchanged.

In [ ]:
from masar.delta_lab import run_maintenance_lab
spark = start_spark(WORK, kafka=False)
try:
    result = run_maintenance_lab(spark, SOURCE, WORK)
    validate_stage_result('lab04b_maintenance', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
    # Observed learning output
    print(json.dumps({'recovery': result['recovery'], 'vacuum': result['vacuum']}, indent=2, default=str))
finally:
    spark.stop()

record_metrics(ROOT, DAY, metrics)

## End-of-lab question

A fare was corrected from SAR 18 to SAR 23. If the same approved revision arrives again, should it become SAR 28? Explain why.

Write your answer and one observed value before opening the model answer. Your reflection is part of the project, worth one point when it explains your own run.

In [ ]:
MY_ANSWER = ""  # Write your own explanation before viewing the model answer.
MY_EVIDENCE = ""  # Include a value you observed and the report or cell that produced it.
reflection_path = record_reflection(ROOT, DAY, 'A fare was corrected from SAR 18 to SAR 23. If the same approved revision arrives again, should it become SAR 28? Explain why.', MY_ANSWER, MY_EVIDENCE)
print("Your reflection:", reflection_path.relative_to(ROOT))
if not MY_ANSWER.strip():
    print("Reflection pending: complete the answer in the file before submitting.")

<details><summary>Model answer: open after your attempt</summary>

The fare remains SAR 23. The revision-aware merge sets the approved value once. An identical revision is a replay, not an instruction to add the SAR 5 difference again.

Keep your first answer. Add a short correction in your reflection file if discussion changed your understanding.

</details>

## Save this day

1. Run the save cell below after all Spark sessions have stopped. It creates two packages.
2. In Colab use **File > Save a copy in GitHub**, select your repository and save this notebook as `day03/STUDENT.ipynb`. Keep your outputs.
3. Download the small `day03_github.zip`, unzip it on your computer, and upload its `student` and updated `day02/dbt` files to the same repository. Use a commit message such as `Complete Day 3`.
4. Keep the full handoff ZIP on Drive for your next session. It includes all cumulative Delta files and the separate Day 2 dbt workspace.
5. Open GitHub and confirm the notebook, reflection and results are visible.

Saving a notebook does not save Colab runtime files. The helper never publishes automatically and never asks you to put a token in a cell.

Before this cell, save your edits to the lab notes, shared project notes and reflection. Files marked **PENDING** still need your own explanation and observed evidence. Do not submit the unchanged prompts as completed work.

In [ ]:
saved = save_day(ROOT, DAY, drive_folder=DRIVE_FOLDER)
print(json.dumps(saved, indent=2))
print("Next: save this executed notebook to your GitHub repository using the File menu.")
if IS_COLAB and not NONINTERACTIVE:
    from google.colab import files
    files.download(saved["github_bundle"])